# Interactive GUI for Price Prediction

Uses the **trained models for prices** and the **SQLite database** maintained by `etl_price.py`.

- **Part 1 — Future Prediction**: forecast tomorrow's hourly price; real prices of yesterday and mean of last 7 days shown for comparison.
- **Part 2 — Historical Comparison**: forecast and actual prices are loaded directly from the pre-computed DB view — no live API calls needed for historical dates.

| Aspect | legacy | ETL |
|---|---|---|
| Models | `*price_lgbm_model.pkl` | `*price_xgb_model.pkl` |
| Historical features | Re-fetched & re-engineered | Read from SQLite DB |
| Historical actuals | SMARD API (filter 4169) | DB `price_de_lu_eur_mwh` |
| wind generation (hist.) | SMARD API (filter 4067, 1225) | DB `gen_wind_onshore_mwh`, `gen_wind_offshore_mwh` |
| solar generation (hist.) | SMARD API (filter 4068) | DB `gen_pv_mwh` |
| conventional generation (hist.) | SMARD API (filter 1227) | DB `gen_other_conventional_mwh` |


## Setup: Imports, Database Update and Model Loading

- Imports all required libraries including `etl_*.py` for DB access
- Calls `update_database()` once (idempotent — completes in seconds if already current)
- Loads the trained models from `../models/`

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('../util'))

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

from datetime import date, timedelta, datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from fetch_demand_data import *
from fetch_price_data import *
from train_predict_model import *
from etl_demand import *
from etl_price import *

# ── Consistent plot style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

# Set European locale so date-picker calendars start on Monday
display(HTML('<script>document.documentElement.lang = "de-DE";</script>'))

# Ensure the database is current (idempotent — usually completes in seconds)
print("Checking / updating database...")
update_database()

# ── Load ETL-trained models ───────────────────────────────────────────────────
_MODEL_PATHS = {
    'LGBM':               '../models/price_lgbm_model.pkl',
    'XGBoost':            '../models/price_xgb_model.pkl',
}
models = {name: load_model_from_pickle(path) for name, path in _MODEL_PATHS.items()}
print("Models loaded:", list(models.keys()))

Checking / updating database...
Series catalog seeded/updated rows: 14

Current SMARD data status:
  price_de_lu_eur_mwh         :  65135 rows | max: 2026-06-06T21:00:00Z
  gen_wind_onshore_mwh        :  65103 rows | max: 2026-06-05T13:00:00Z
  gen_wind_offshore_mwh       :  65104 rows | max: 2026-06-05T14:00:00Z
  gen_pv_mwh                  :  65104 rows | max: 2026-06-05T14:00:00Z
  gen_other_conventional_mwh  :  65105 rows | max: 2026-06-05T15:00:00Z
  forecast_wind_onshore_mwh   :  65135 rows | max: 2026-06-06T21:00:00Z
  forecast_wind_offshore_mwh  :  65135 rows | max: 2026-06-06T21:00:00Z
  forecast_pv_mwh             :  65135 rows | max: 2026-06-06T21:00:00Z

Fetching and storing SMARD price/generation series: 2026-06-05 → 2026-06-10
Batch ingestion result: {'run_id': 27, 'status': 'success', 'total_inserted': 647, 'per_series': {'price_de_lu_eur_mwh': 72, 'gen_wind_onshore_mwh': 90, 'gen_wind_offshore_mwh': 90, 'gen_pv_mwh': 90, 'gen_other_conventional_mwh': 89, 'forecast_wind

---
# Part 1 — Future Prediction

- **Predict for Tomorrow** — forecast the full next day (00:00–23:00 Berlin time)

Tomorrow's features are built live: energy lag context is fetched from SMARD (last ~15 days) and weather forecast is fetched from the Open-Meteo API.  
Energy lag column names are remapped from the legacy `fetch_prepare_data` naming to the ETL DB naming so they match the feature set the model was trained on.

Results are shown as a line plot **and** an hourly table side-by-side.

In [2]:
FILTER_SMARD_FORECAST = 411

future_model_dd = widgets.Dropdown(
    options=list(models.keys()),
    value='LGBM',
    description='Model:',
    layout=widgets.Layout(width='240px'),
)

now_label = widgets.HTML(
    value=f'<b>Now (Berlin):</b> {pd.Timestamp.now(tz="Europe/Berlin").strftime("%Y-%m-%d %H:%M")}',
)

btn_tomorrow = widgets.Button(
    description='Predict for Tomorrow',
    button_style='primary',
    icon='arrow-right',
    layout=widgets.Layout(width='210px'),
)

status_future = widgets.Label(value='Ready.')
output_future = widgets.Output()


def _strip_tz(series):
    # Convert tz-aware timestamps to timezone-naive Europe/Berlin for matplotlib rendering.
    return series.dt.tz_convert('Europe/Berlin').dt.tz_localize(None)


def _render_future(df_future, predictions, model_name, title,
                   x_fmt='%H:%M', x_label='Hour (Europe/Berlin)', df_smard_fc=None):
    fig, (ax_plot, ax_tbl) = plt.subplots(
        1, 2, figsize=(14, 5),
        gridspec_kw={'width_ratios': [2.5, 1]},
    )
    if df_smard_fc is not None and not df_smard_fc.empty:
        ax_plot.plot(_strip_tz(df_smard_fc['time']), df_smard_fc['EnergyDemand'],
                     color='mediumseagreen', linewidth=1.5, linestyle='-.',
                     label='SMARD official forecast')
    ax_plot.plot(_strip_tz(df_future['time']), predictions,
                 linewidth=2, color='darkorange', linestyle='--',
                 label=f'{model_name} prediction')
    ax_plot.xaxis.set_major_formatter(mdates.DateFormatter(x_fmt))
    ax_plot.set_xlabel(x_label)
    ax_plot.set_ylabel('Energy Demand (MWh)')
    ax_plot.set_title(title)
    ax_plot.legend()
    ax_plot.grid(True, alpha=0.3)
    fig.autofmt_xdate()

    df_tbl = df_future[['time']].copy()
    df_tbl['predicted_MWh'] = predictions.round(0).astype(int)
    df_tbl['Date / Hour (Berlin)'] = _strip_tz(df_tbl['time']).dt.strftime('%m-%d %H:%M')
    df_tbl = df_tbl[['Date / Hour (Berlin)', 'predicted_MWh']].reset_index(drop=True)

    ax_tbl.axis('off')
    tbl = ax_tbl.table(
        cellText=df_tbl.values,
        colLabels=df_tbl.columns,
        loc='center',
        cellLoc='right',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.1)
    plt.tight_layout()
    plt.show()


def on_tomorrow_clicked(_b):
    now_label.value = (
        f'<b>Now (Berlin):</b> {pd.Timestamp.now(tz="Europe/Berlin").strftime("%Y-%m-%d %H:%M")}'
    )
    tomorrow_str = (date.today() + timedelta(days=1)).isoformat()
    model_name   = future_model_dd.value

    with output_future:
        clear_output(wait=True)

        # 1. Fetch SMARD official forecast for tomorrow ────────────────────────
        status_future.value = 'Fetching SMARD official forecast for tomorrow...'
        try:
            df_smard_fc = fetch_smard_netzlast(
                str(date.today()), tomorrow_str, filter_id=FILTER_SMARD_FORECAST,
            )
            if not df_smard_fc.empty:
                _tomorrow_berlin = pd.Timestamp(tomorrow_str, tz='Europe/Berlin')
                df_smard_fc = df_smard_fc[
                    df_smard_fc['time'] >= _tomorrow_berlin
                ].reset_index(drop=True)
        except Exception:
            df_smard_fc = pd.DataFrame(columns=['time', 'EnergyDemand'])

        status_future.value = (
            'SMARD forecast not yet published — ML only.'
            if df_smard_fc.empty else
            f'SMARD forecast: {len(df_smard_fc)} hours fetched.'
        )

        # 2. Build features using ETL approach:
        #    - energy lag context loaded from DB (last 168 rows, no SMARD API call)
        #    - weather forecast fetched live from Open-Meteo API
        #    Column names already match ETL model schema — no rename needed.
        status_future.value = f'Preparing features for {tomorrow_str} (energy context from DB)...'
        try:
            df_future = prepare_for_prediction_tomorrow_etl(tomorrow_str)
        except Exception as exc:
            status_future.value = f'Feature preparation failed: {exc}'
            return
        if df_future.empty:
            status_future.value = 'No features returned — check API connectivity.'
            return

        X     = df_future.drop(columns=['time'], errors='ignore')
        model = models[model_name]
        if hasattr(model, 'feature_names_in_'):
            X = X.reindex(columns=model.feature_names_in_)
        preds = model.predict(X)

        _render_future(
            df_future, preds, model_name,
            title=f'Energy Demand Forecast — Tomorrow ({tomorrow_str})  [{model_name}]',
            df_smard_fc=df_smard_fc if not df_smard_fc.empty else None,
        )
        status_future.value = f'Done — {tomorrow_str} ({model_name})'


btn_tomorrow.on_click(on_tomorrow_clicked)

display(widgets.VBox([
    now_label,
    widgets.HBox(
        [future_model_dd, btn_tomorrow, status_future],
        layout=widgets.Layout(align_items='center', gap='12px'),
    ),
    output_future,
]))

---
# Part 2 — Historical Comparison: Actual vs SMARD Forecast vs ML Prediction

Select a **From** and **To** date to compare three demand series side by side:
- **Actual** — demand stored in the DB (`energy_demand_mwh`)
- **SMARD official forecast** — stored in the DB (`smard_forecast_mwh`)
- **ML Prediction** — model prediction from the pre-computed feature columns in the DB

**Key advantage over the legacy notebook**: a single DB query replaces three API calls (SMARD actual, SMARD forecast, energy for lag features) plus weather re-fetch. Historical comparisons are therefore much faster.

- Maximum range: **1 year**
- Calendar week starts on **Monday** (European style)

In [ ]:
MAX_RANGE_DAYS = 365

_default_to   = date.today() - timedelta(days=1)
_default_from = _default_to - timedelta(days=6)

hist_model_dd = widgets.Dropdown(
    options=list(models.keys()),
    value='LGBM',
    description='Model:',
    layout=widgets.Layout(width='220px'),
)

date_from = widgets.DatePicker(
    description='From:',
    value=_default_from,
    min=date(2019, 1, 8),    # DB starts at 2019-01-08 (after lag warm-up)
    max=date.today() - timedelta(days=1),
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px'),
)

date_to = widgets.DatePicker(
    description='To:',
    value=_default_to,
    min=date(2019, 1, 8),
    max=date.today() - timedelta(days=1),
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px'),
)

range_warning = widgets.HTML(value='')

btn_compare = widgets.Button(
    description='Compare Prediction vs Actual',
    button_style='warning',
    icon='bar-chart',
    layout=widgets.Layout(width='270px'),
)

status_hist = widgets.Label(value='Ready.')
output_hist = widgets.Output()


# ── Live range validation ─────────────────────────────────────────────────────
def _validate_range(_change=None):
    try:
        delta = (date_to.value - date_from.value).days
        if delta < 0:
            range_warning.value = (
                '<span style="color:red;font-weight:bold;">'
                '&#9888; "To" date must be on or after "From" date.</span>'
            )
            btn_compare.disabled = True
        elif delta > MAX_RANGE_DAYS:
            range_warning.value = (
                f'<span style="color:orange;font-weight:bold;">'
                f'&#9888; Range is {delta} days — max is {MAX_RANGE_DAYS}. '
                f'Please narrow the selection.</span>'
            )
            btn_compare.disabled = True
        else:
            range_warning.value = (
                f'<span style="color:green;">Range: {delta + 1} day(s)  &#10003;</span>'
            )
            btn_compare.disabled = False
    except Exception:
        range_warning.value = ''


date_from.observe(_validate_range, names='value')
date_to.observe(_validate_range, names='value')
_validate_range()


# ── Compare button handler ────────────────────────────────────────────────────
def on_compare_clicked(_b):
    from_str   = str(date_from.value)
    to_str     = str(date_to.value)
    model_name = hist_model_dd.value

    with output_hist:
        clear_output(wait=True)

        # 1. Single DB query: features + actual demand + SMARD forecast ────────
        status_hist.value = f'Loading from DB: {from_str} to {to_str}...'
        conn = get_connection()
        try:
            df_db = load_combined_data(conn, start_date=from_str, end_date=to_str)
        finally:
            conn.close()

        if df_db.empty:
            status_hist.value = 'No data in DB for the selected range.'
            return

        # 2. Extract actual demand and SMARD forecast ─────────────────────────
        s_actual = df_db.set_index('time')['energy_demand_mwh'].rename('Actual')
        s_smard  = (
            df_db.set_index('time')['smard_forecast_mwh'].rename('SMARD Forecast')
            if 'smard_forecast_mwh' in df_db.columns
            else pd.Series(dtype=float, name='SMARD Forecast')
        )

        # 3. Build feature matrix and predict ─────────────────────────────────
        status_hist.value = f'Running {model_name}...'
        _drop = ['time', 'energy_demand_mwh', 'smard_forecast_mwh', 'data_source']
        X = df_db.drop(columns=[c for c in _drop if c in df_db.columns])
        model = models[model_name]
        if hasattr(model, 'feature_names_in_'):
            X = X.reindex(columns=model.feature_names_in_)
        preds  = model.predict(X)
        s_pred = pd.Series(preds, index=df_db['time'], name='ML Prediction')

        # 4. Combine and plot ─────────────────────────────────────────────────
        df_plot = pd.concat([s_actual, s_smard, s_pred], axis=1)

        days_in_range = (date_to.value - date_from.value).days + 1
        if days_in_range <= 3:
            x_fmt = '%m-%d %H:%M'
        elif days_in_range <= 31:
            x_fmt = '%Y-%m-%d'
        else:
            x_fmt = '%Y-%m'

        df_plot.index = df_plot.index.tz_convert('Europe/Berlin').tz_localize(None)
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.plot(df_plot.index, df_plot['Actual'],
                color='steelblue', linewidth=1.5, label='Actual (DB)')
        if df_plot['SMARD Forecast'].notna().any():
            ax.plot(df_plot.index, df_plot['SMARD Forecast'],
                    color='mediumseagreen', linewidth=1.5, linestyle='-.',
                    label='SMARD official forecast (DB)')
        ax.plot(df_plot.index, df_plot['ML Prediction'],
                color='darkorange', linewidth=1.5, linestyle='--',
                label=f'ML Prediction ({model_name})')
        ax.xaxis.set_major_formatter(mdates.DateFormatter(x_fmt))
        ax.set_xlabel('Date / Time (Europe/Berlin)')
        ax.set_ylabel('Energy Demand (MWh)')
        ax.set_title(
            f'Energy Demand — Actual vs SMARD Forecast vs ML Prediction '
            f'from {from_str} to {to_str}  [{model_name}]'
        )
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()

        # 5. Metrics table ─────────────────────────────────────────────────────
        df_ml_cmp = df_plot[['Actual', 'ML Prediction']].dropna()
        mae_ml    = (df_ml_cmp['Actual'] - df_ml_cmp['ML Prediction']).abs().mean()
        rmse_ml   = ((df_ml_cmp['Actual'] - df_ml_cmp['ML Prediction']) ** 2).mean() ** 0.5

        metrics_html = (
            '<table style="border-collapse:collapse;font-family:monospace;font-size:13px;">'
            '<tr style="background:#f0f0f0;">'
            '<th style="padding:4px 16px;border:1px solid #ccc;">Series</th>'
            '<th style="padding:4px 16px;border:1px solid #ccc;">MAE (MWh)</th>'
            '<th style="padding:4px 16px;border:1px solid #ccc;">RMSE (MWh)</th>'
            '<th style="padding:4px 16px;border:1px solid #ccc;">Points</th></tr>'
            f'<tr><td style="padding:4px 16px;border:1px solid #ccc;">'
            f'ML Prediction ({model_name})</td>'
            f'<td style="padding:4px 16px;border:1px solid #ccc;text-align:right;">'
            f'{mae_ml:,.0f}</td>'
            f'<td style="padding:4px 16px;border:1px solid #ccc;text-align:right;">'
            f'{rmse_ml:,.0f}</td>'
            f'<td style="padding:4px 16px;border:1px solid #ccc;text-align:right;">'
            f'{len(df_ml_cmp)}</td></tr>'
        )
        if df_plot['SMARD Forecast'].notna().any():
            df_smard_cmp = df_plot[['Actual', 'SMARD Forecast']].dropna()
            mae_smard    = (df_smard_cmp['Actual'] - df_smard_cmp['SMARD Forecast']).abs().mean()
            rmse_smard   = (
                (df_smard_cmp['Actual'] - df_smard_cmp['SMARD Forecast']) ** 2
            ).mean() ** 0.5
            metrics_html += (
                '<tr><td style="padding:4px 16px;border:1px solid #ccc;">'
                'SMARD official forecast</td>'
                f'<td style="padding:4px 16px;border:1px solid #ccc;text-align:right;">'
                f'{mae_smard:,.0f}</td>'
                f'<td style="padding:4px 16px;border:1px solid #ccc;text-align:right;">'
                f'{rmse_smard:,.0f}</td>'
                f'<td style="padding:4px 16px;border:1px solid #ccc;text-align:right;">'
                f'{len(df_smard_cmp)}</td></tr>'
            )
        metrics_html += '</table>'
        display(widgets.HTML(metrics_html))

        status_hist.value = f'Done — {from_str} to {to_str} ({model_name})'


btn_compare.on_click(on_compare_clicked)

display(widgets.VBox([
    widgets.HBox(
        [date_from, date_to, hist_model_dd],
        layout=widgets.Layout(align_items='flex-end', gap='12px'),
    ),
    widgets.HBox(
        [btn_compare, status_hist],
        layout=widgets.Layout(align_items='center', gap='12px'),
    ),
    range_warning,
    output_hist,
]))